# Context engineering

The model sees the messages we send on the current call. That message list is its context.

In this notebook, we will:

- measure context growth;
- move large text out of the message list;
- use a sub-agent as a context boundary;
- compact old messages; and
- choose the right technique for each kind of overflow.

In [ ]:
import json
import os
from getpass import getpass
from pathlib import Path

import litellm
import pandas as pd
from dotenv import load_dotenv

course_root = Path.cwd()
if not (course_root / "pyproject.toml").exists():
    course_root = course_root.parent
load_dotenv(course_root / ".env")
if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("OpenRouter API key: ")

MODEL = os.getenv("MODEL", "openrouter/deepseek/deepseek-v3.2")
litellm.drop_params = True
litellm.cache = litellm.Cache(type="disk", disk_cache_dir=str(course_root / ".litellm_cache"))

## 1. Measure growth

A flat agent loop sends the full message history on every step. The history grows, so later calls cost more tokens than earlier calls.

In [ ]:
from shoplab.context import ContextLedger, count_tokens

messages = [{"role": "system", "content": "You are a shop assistant."}]
context_log = ContextLedger(model=MODEL)

for step in range(1, 7):
    messages.append({"role": "user", "content": "Question " + str(step) + ": " + "parcel details " * 30})
    messages.append({"role": "assistant", "content": "Answer " + str(step) + ": " + "checked details " * 20})
    context_log.observe(messages, label=f"step {step}")

pd.DataFrame(context_log.growth())

The final context size is not the whole bill. Each earlier call also sent its own growing history. Adding all step sizes shows why a long flat loop can become expensive quickly.

In [ ]:
step_tokens = [row["tokens"] for row in context_log.growth()]

print("tokens in final context:", step_tokens[-1])
print("tokens sent across all steps:", sum(step_tokens))
print("peak:", context_log.peak())

## 2. Offload large text

A large plan or tool result does not need to stay in every model call. We can write it to a file and keep a small handle in context.

In [ ]:
from shoplab.context import load_offload, offload

large_result = "Policy search result. " * 500
before = count_tokens(large_result, model=MODEL)

handle = offload(
    large_result,
    tag="simplified-policy",
    dir=course_root / "artifacts/offload",
)
after = count_tokens(json.dumps(handle), model=MODEL)

print("full text tokens:", before)
print("handle tokens:", after)
print("handle:", handle)

In [ ]:
restored = load_offload(handle)

print("restored characters:", len(restored))
print("same content:", restored == large_result)

Offloading saves context, but the model cannot use the full text until a tool loads it again. A handle is a reference, not memory.

## 3. Use a sub-agent as a context boundary

A sub-agent gets a fresh message list. The parent receives only the compact result, not every tool call and observation.

In [ ]:
from shoplab.context import run_subagent
from shoplab.tools import standard_tools

policy_worker = run_subagent(
    "Find the policy for an opened item returned within the normal window.",
    {name: standard_tools()[name] for name in ("search_policy",)},
    model=MODEL,
    system="You are a policy researcher. Return one short finding.",
    max_steps=4,
)

print(policy_worker)

The parent sees answer, steps, and stop_reason. The worker's intermediate conversation does not enter the parent's context.

## 4. Compact old conversation

Compaction keeps the latest messages and replaces older middle messages with one summary.

In [ ]:
from shoplab.context import compact

long_chat = [{"role": "system", "content": "You handle shop tickets."}]
for number in range(1, 9):
    long_chat.append({"role": "user", "content": f"Ticket {number}: " + "details " * 25})
    long_chat.append({"role": "assistant", "content": f"Result {number}: " + "decision notes " * 20})

short_chat = compact(long_chat, keep_last=4)

print("before:", len(long_chat), "messages,", count_tokens(long_chat, model=MODEL), "tokens")
print("after: ", len(short_chat), "messages,", count_tokens(short_chat, model=MODEL), "tokens")
print("summary message:", short_chat[1]["content"][:180])

Compaction is lossy. Details left out of the summary are gone from the model's view. Keep exact facts elsewhere when they must not be lost.

## Choose the smallest useful technique

| Problem | Technique |
|---|---|
| The task is long and unclear | Write a short plan |
| One result is very large | Offload it and keep a handle |
| A side task has many private steps | Use a sub-agent |
| Old conversation is filling the window | Compact the older middle |

## What to remember

1. Count context tokens instead of guessing.
2. A flat loop resends its growing history.
3. Offloading trades immediate access for a small handle.
4. A sub-agent keeps its working context private.
5. Compaction saves space by losing detail.
6. Choose the smallest technique that solves the actual overflow.

## Small practice steps

1. Add two more turns to the growth example.
2. Offload a different large string and load it back.
3. Change keep_last from four to two and compare token counts.